<a href="https://colab.research.google.com/github/GretelKMendez/Tareas-Mac-IA/blob/main/VidegameSales.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Identificación del Problema y Variable Objetivo
Tipo de problema: Clasificación Binaria.

Variable Objetivo: Creamos la variable Es_Exitoso.

Definición: Un juego se considera "exitoso" (1) si sus ventas globales (Global_Sales) son mayores a 1 millón de copias. Si es menor o igual, se considera no exitoso (0).

Justificación de variables (Features):

Usar: Platform, Genre y Publisher. Son datos conocidos antes del lanzamiento y determinan el mercado potencial.

No usar: Rank, NA_Sales, EU_Sales, JP_Sales, Other_Sales. Estas variables contienen información del futuro (ventas ya realizadas); usarlas sería "trampa" (Data Leakage). Name se excluye por tener demasiados valores únicos.

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# 1. Carga de datos
df = pd.read_csv('vgsales.csv')

# 2. Manejo de valores nulos
# Los años y distribuidores faltantes son pocos, los eliminamos para no inventar datos.
df = df.dropna(subset=['Year', 'Publisher'])

# 3. Creación de la variable objetivo
# Definimos éxito como ventas > 1 millón
df['Es_Exitoso'] = (df['Global_Sales'] > 1.0).astype(int)

# 4. Selección de variables funcionales
# Solo tomamos lo que sabemos antes de que el juego salga a la venta
features = ['Platform', 'Genre', 'Publisher']
X = df[features].copy()
y = df['Es_Exitoso']

# 5. Codificación de variables categóricas
# Los modelos no entienden texto ("Action", "Nintendo"), así que lo convertimos a números.
le = LabelEncoder()
for col in features:
    X[col] = le.fit_transform(X[col].astype(str))

# 6. División de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print(df)

        Rank                                              Name Platform  \
0          1                                        Wii Sports      Wii   
1          2                                 Super Mario Bros.      NES   
2          3                                    Mario Kart Wii      Wii   
3          4                                 Wii Sports Resort      Wii   
4          5                          Pokemon Red/Pokemon Blue       GB   
...      ...                                               ...      ...   
16593  16596                Woody Woodpecker in Crazy Castle 5      GBA   
16594  16597                     Men in Black II: Alien Escape       GC   
16595  16598  SCORE International Baja 1000: The Official Game      PS2   
16596  16599                                        Know How 2       DS   
16597  16600                                  Spirits & Spells      GBA   

         Year         Genre   Publisher  NA_Sales  EU_Sales  JP_Sales  \
0      2006.0        Sport

In [4]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

# --- MODELO 1: Árbol de Decisión ---
modelo_arbol = DecisionTreeClassifier(max_depth=10, random_state=42)
modelo_arbol.fit(X_train, y_train)
pred_arbol = modelo_arbol.predict(X_test)

# --- MODELO 2: Bosques Aleatorios ---
modelo_bosque = RandomForestClassifier(n_estimators=100, random_state=42)
modelo_bosque.fit(X_train, y_train)
pred_bosque = modelo_bosque.predict(X_test)

# --- COMPARACIÓN ---
print(f"Precisión Árbol de Decisión: {accuracy_score(y_test, pred_arbol):.4f}")
print(f"Precisión Bosque Aleatorio: {accuracy_score(y_test, pred_bosque):.4f}")

Precisión Árbol de Decisión: 0.8795
Precisión Bosque Aleatorio: 0.8740


Accuracy (Precisión Global): Nos dirá qué porcentaje de juegos clasificamos correctamente.
Para saber si un modelo de Machine Learning es "bueno", no existe un número mágico universal, ya que todo depende del contexto. Sin embargo, aquí te doy las reglas de oro para interpretar ese valor:

. El umbral estándar
En términos generales, buscamos que la precisión (Accuracy) esté lo más cerca posible de 1.0 (o 100%).

0.70 a 0.80 (70% - 80%): Se considera un modelo aceptable o decente.